In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

dbutils.widgets.text("catalogo", "proyecto_ecommerce")
catalogo = dbutils.widgets.get("catalogo")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalogo}.gold")

df_clientes = spark.table(f"{catalogo}.silver.clientes")
df_productos = spark.table(f"{catalogo}.silver.productos")
df_ordenes = spark.table(f"{catalogo}.silver.ordenes")

print(f"Clientes: {df_clientes.count()}")
print(f"Productos: {df_productos.count()}")
print(f"Órdenes: {df_ordenes.count()}")

##GENERAR DIM FECHAS

In [0]:
fechas_distintas = df_ordenes.select("fecha").distinct()

MESES_ES = {
    1: "Enero", 2: "Febrero", 3: "Marzo", 4: "Abril", 5: "Mayo", 6: "Junio",
    7: "Julio", 8: "Agosto", 9: "Setiembre", 10: "Octubre", 11: "Noviembre", 12: "Diciembre",
}
mes_udf = F.udf(lambda m: MESES_ES[m])

DIAS_ES = {
    "Monday": "Lunes", "Tuesday": "Martes", "Wednesday": "Miercoles",
    "Thursday": "Jueves", "Friday": "Viernes", "Saturday": "Sabado", "Sunday": "Domingo",
}
dia_udf = F.udf(lambda d: DIAS_ES[d])

dim_tiempo = (
    fechas_distintas
    .withColumn("fecha_id", F.date_format("fecha", "yyyyMMdd").cast("int"))
    .withColumn("anio", F.year("fecha"))
    .withColumn("mes", F.month("fecha"))
    .withColumn("nombre_mes", mes_udf(F.col("mes")))
    .withColumn("trimestre", F.quarter("fecha"))
    .withColumn("dia_semana", dia_udf(F.date_format("fecha", "EEEE")))
    .select("fecha_id", "fecha", "anio", "mes", "nombre_mes", "trimestre", "dia_semana")
)

display(dim_tiempo.orderBy("fecha").limit(10))
print(f"gold.dim_tiempo (aún sin guardar) -> {dim_tiempo.count()} filas")

In [0]:
(dim_tiempo.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.gold.dim_tiempo"))

print(f"Guardado: {catalogo}.gold.dim_tiempo -> {dim_tiempo.count()} filas")

## GENERAR DIM CLIENTES

In [0]:
dim_cliente = df_clientes.select(
    "cliente_id", "nombre_cliente", "segmento", "region", "fecha_registro"
)

display(dim_cliente.limit(10))
print(f"gold.dim_cliente (aún sin guardar) -> {dim_cliente.count()} filas")

In [0]:
(dim_cliente.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.gold.dim_cliente"))

print(f"Guardado: {catalogo}.gold.dim_cliente -> {dim_cliente.count()} filas")

## GENERAR DIM PRODUCTO

In [0]:
dim_producto = df_productos.select(
    "producto_id", "nombre_producto", "categoria", "marca", "precio_referencia"
)

display(dim_producto)
print(f"gold.dim_producto (aún sin guardar) -> {dim_producto.count()} filas")

In [0]:
(dim_producto.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.gold.dim_producto"))

print(f"Guardado: {catalogo}.gold.dim_producto -> {dim_producto.count()} filas")

## GENERAR DIM REGION

In [0]:
dim_region = (
    df_clientes.select("region").distinct()
    .withColumn("region_id", F.row_number().over(Window.orderBy("region")))
    .select("region_id", "region")
)

display(dim_region)
print(f"gold.dim_region (aún sin guardar) -> {dim_region.count()} filas")

In [0]:
(dim_region.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.gold.dim_region"))

print(f"Guardado: {catalogo}.gold.dim_region -> {dim_region.count()} filas")

In [0]:
df_gold_cliente = spark.table(f"{catalogo}.gold.dim_cliente")

ids_huerfanos = (
    df_ordenes.select("cliente_id").distinct()
    .join(df_gold_cliente.select("cliente_id"), "cliente_id", "left_anti")
)

ordenes_huerfanas = df_ordenes.join(ids_huerfanos, "cliente_id").count()
print(f"Órdenes con cliente_id que no existe en dim_cliente: {ordenes_huerfanas}")

## GENERAR TABLA FACT VENTAS

In [0]:
df_gold_region = spark.table(f"{catalogo}.gold.dim_region")

fact_ventas = (
    df_ordenes
    .join(df_clientes.select("cliente_id", "region"), "cliente_id", "inner")  # excluye los 3 huérfanos
    .join(df_gold_region, "region", "inner")
    .withColumn("fecha_id", F.date_format("fecha", "yyyyMMdd").cast("int"))
    .withColumn("descuento_monto", F.round(F.col("monto_bruto") - F.col("monto_neto"), 2))
    .select(
        "orden_id", "fecha_id", "cliente_id", "producto_id", "region_id",
        "canal", "cantidad", "precio_unitario", "monto_bruto",
        "descuento_monto", "monto_neto",
    )
)

display(fact_ventas.limit(10))

total_fact = fact_ventas.count()
print(f"gold.fact_ventas (aún sin guardar) -> {total_fact} filas")
print(f"Diferencia vs. silver.ordenes ({df_ordenes.count()}): {df_ordenes.count() - total_fact} excluidas por cliente_id inválido")

In [0]:
(fact_ventas.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.gold.fact_ventas"))

print(f"Guardado: {catalogo}.gold.fact_ventas -> {fact_ventas.count()} filas")